# 01 — Exploratory Data Analysis
This notebook explores the raw Amazon electronics review dataset before any modelling. We look at rating distributions, review lengths, brand coverage, and data quality.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"

## 1. Load Raw Data

In [ ]:
files = [
    "Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv",
    "Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv"
]

dfs = []
for f in files:
    path = os.path.join(RAW_DIR, f)
    df = pd.read_csv(path, low_memory=False)
    dfs.append(df)
    print(f"{f}: {len(df):,} rows")

raw = pd.concat(dfs, ignore_index=True)
print(f"\nTotal combined rows: {len(raw):,}")
print(f"Columns: {list(raw.columns)}")

## 2. Preview & Basic Info

In [ ]:
raw.head(3)

In [ ]:
# Columns we care about
cols = ["name", "brand", "reviews.rating", "reviews.text", "reviews.title", "reviews.date"]
subset = raw[cols].copy()
subset.columns = ["product", "brand", "rating", "review_text", "review_title", "review_date"]

print("Missing values:")
print(subset.isnull().sum())
print(f"\nDuplicate reviews: {subset.duplicated(subset=['review_text']).sum():,}")

## 3. Load Cleaned Data

In [ ]:
df = pd.read_csv(os.path.join(PROCESSED_DIR, "reviews_cleaned.csv"), parse_dates=["review_date"])
print(f"Cleaned dataset: {len(df):,} rows")
df.head(3)

## 4. Rating Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
rating_counts = df["rating"].value_counts().sort_index()
bars = ax.bar(rating_counts.index, rating_counts.values, color=["#e74c3c","#e67e22","#f1c40f","#2ecc71","#27ae60"])
ax.set_xlabel("Star Rating")
ax.set_ylabel("Number of Reviews")
ax.set_title("Rating Distribution")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f"{int(bar.get_height()):,}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

print(rating_counts.to_string())

## 5. Top Brands

In [ ]:
top_brands = df["brand"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(top_brands.index[::-1], top_brands.values[::-1], color="#3498db")
ax.set_xlabel("Number of Reviews")
ax.set_title("Top 10 Brands by Review Count")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

## 6. Review Length Distribution

In [ ]:
df["review_length"] = df["review_text"].str.len()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["review_length"].clip(upper=2000), bins=50, color="#9b59b6", edgecolor="white")
ax.set_xlabel("Review Length (characters, capped at 2000)")
ax.set_ylabel("Count")
ax.set_title("Review Length Distribution")
plt.tight_layout()
plt.show()

print(df["review_length"].describe().apply(lambda x: f"{x:,.0f}"))

## 7. Reviews Over Time

In [ ]:
monthly = df.set_index("review_date").resample("ME").size()
monthly = monthly[monthly.index.year >= 2015]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly.index, monthly.values, color="#e67e22", linewidth=2)
ax.fill_between(monthly.index, monthly.values, alpha=0.2, color="#e67e22")
ax.set_xlabel("Date")
ax.set_ylabel("Reviews per Month")
ax.set_title("Review Volume Over Time")
plt.tight_layout()
plt.show()

## 8. Summary Stats

In [ ]:
print(f"Total reviews (cleaned): {len(df):,}")
print(f"Unique products:         {df['product'].nunique():,}")
print(f"Unique brands:           {df['brand'].nunique():,}")
print(f"Average rating:          {df['rating'].mean():.2f} / 5")
print(f"Date range:              {df['review_date'].min().date()} → {df['review_date'].max().date()}")